此文件描述來源擴充前的整合快照。輸出／讀取位置已改為 snapshots/pre_source_expansion_20261019，以保留當時bytes，避免重跑舊版覆寫目前current。最新資料由 expand_candidate_sources_20261019.ipynb 產生。

# README
- **Author**: RUEI-HSIANG, CHANG (B12303030)，Codex 協助
- **Created At**: 2026-10-19
- **Last Modified At**: 2026-10-19

## What does this file do?
將先前已支持身分連結的690位議員2022官方申報生日／性別及非首次參選證據接回目前清單；另建全1,502人背景表及未完成身分核對的歷史命中表。姓名＋縣市歷史命中本身不升級為已核實。一般經歷原文與行政職不同；舊版全部保留。

## What does this file take?
程式PINS逐檔固定：round4主表／欄位表、692筆身分審核、28筆競選人物頁、背景／社群證據、原692頁名單、中選會archive及六份2022議員候選人原始表／選區elbase。逐一檢查已支持者的原始列姓名、生日及性別與審核表一致，民國生日須可轉為真實西元日期。

## What does this file output?
- data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv：1,502位議員背景與來源狀態，一人一列。
- data/processed/council_history_pending_identity_20261019.csv：同名同縣市、仍待身分核對的2022紀錄；不作已核實屬性值。
- data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv：目前1,583人狀態入口。
- data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv：目前31,660個欄位與來源入口。
文件：data/processed/current_inventory_20261019_documentation.ipynb。來源年2022、身分連結及生日查核支持程度分開保存，不宣稱戶籍認證或2026現任；不以歷史未命中填首次=1。這次開始提供固定名稱current入口，保留舊日期／round版本追溯。


In [1]:

from pathlib import Path
import json,hashlib,csv,io,re,unicodedata,datetime
from collections import defaultdict
import pandas as pd
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'AGENTS.md').exists() and (p/'templates').is_dir())
def sha(p):return hashlib.sha256(p.read_bytes()).hexdigest()
PINS={'data/processed/candidate_master_reviewed_round4_20261019.csv': '8af6660bfdee31ab3f2f0189218fbc607b3d779f0c2d349575fe1d3909daa0fd', 'data/processed/candidate_fields_reviewed_round4_20261019.csv': 'eb36a93dbe7e38e2a7dbad48741b1f7f37eea114354ca18d021951acfb30fb74', 'data/processed/council_identity_education_review_20261019.csv': 'becae07e111c6f99203902c0759dd12c7b7f9d21e029e961e24c12bf6467524e', 'data/processed/council_campaign_profiles_20261019.csv': '28fed78971de8248fab91ce32c889851ae2df17de303c2ea25b3e0bb9778ae00', 'data/processed/candidate_background_evidence_round3_20261019.csv': '5195c8b1902f8228f8001f3986e7d2d433b0662e2d5a069bdd015a491d4bf663', 'data/processed/candidate_social_account_evidence_20261019.csv': 'b15e1bf88876de049533f381771920c3c560e18eef5c53e04fb8142cbedc5ff9', 'data/processed/candidate_profile_leads_20261019.csv': '52b723bdeea132df0f5ef9f9ac97594993784ba84edb796c82f0959d63236a3f', 'data/raw/cec_all_offices_manifest.json': '9cc85f7f913f5de717ba92ec7698c73cedb29df506fba59a69933d1ac3517231', 'data/raw/council_review_20261019/manifest.json': '05d2b4640f129796795621b0b0962f903f1c20b83240e6e511f7f5da49fb7457', 'data/raw/cec_all_offices/106_elcand.csv': '424fe34e9450796a3a2235ea75f0ff245104a6fa2f586a6bdd269c0e14ba6104', 'data/raw/cec_all_offices/107_elcand.csv': 'af38d1e0ca691d9e353fcfb3e154f0cbe1f178d6e796ce7441ef7cb52d02f1b9', 'data/raw/cec_all_offices/108_elcand.csv': '42cedd03d0943b4340eb69f85747c988be042eeb0321418a932c2d92d6340517', 'data/raw/cec_all_offices/109_elcand.csv': 'd5dc7b0ad7c5bb2b9457e1169c8afbf9c04383bbd7dd3eb771cb73062165c93e', 'data/raw/cec_all_offices/110_elcand.csv': 'b323d6713df428b57161e3b1b6799ebf25d679ef9948e8fe4fbb3fa0d7cfbc73', 'data/raw/cec_all_offices/111_elcand.csv': '231579c3e82e80876f97ca48c9dba024f83691dd0e93f48c06cdc7875fe68a3e', 'data/raw/council_review_20261019/T1_city_elbase.csv': '59c702f5384e839a5665c15eaf029efd660ade19924d09f9d4d57b7d917c1eef', 'data/raw/council_review_20261019/T1_prv_elbase.csv': '8320dd2af356191191e3eff5c27bceb7219b8432ef439d33e8215ec01f1b56ec', 'data/raw/council_review_20261019/T2_city_elbase.csv': 'fd87beb45b9406deab03446e7b4b2a079c1dc31ee310f384272faefa9a2674f0', 'data/raw/council_review_20261019/T2_prv_elbase.csv': '23ff2c352fcdfd7024f0c3eeda56d6da2c98cb3027fba460ca547c8ac37254e1', 'data/raw/council_review_20261019/T3_city_elbase.csv': '163487ae4d04748de0d00ebe6e5b79f982852d8312f13447a8d2a13724b037bb', 'data/raw/council_review_20261019/T3_prv_elbase.csv': 'ca4d0b75424876a61dee226cf94d3d2277e5af22cdde20390369d0fc7e1de4b1'}
for p,h in PINS.items():assert sha(ROOT/p)==h,p
def read(p):return pd.read_csv(ROOT/p,dtype=str).fillna('')
def norm(s):return re.sub(r'\s+','',unicodedata.normalize('NFKC',s)).replace('台','臺').replace('温','溫')
master=read('data/processed/candidate_master_reviewed_round4_20261019.csv')
fields=read('data/processed/candidate_fields_reviewed_round4_20261019.csv')
review=read('data/processed/council_identity_education_review_20261019.csv')
campaign=read('data/processed/council_campaign_profiles_20261019.csv')
bg=read('data/processed/candidate_background_evidence_round3_20261019.csv')
social=read('data/processed/candidate_social_account_evidence_20261019.csv')
old_profiles=read('data/processed/candidate_profile_leads_20261019.csv')
assert len(master)==1583 and master.candidate_key.is_unique and len(fields)==31660
am=json.loads((ROOT/'data/raw/cec_all_offices_manifest.json').read_text())
gm=json.loads((ROOT/'data/raw/council_review_20261019/manifest.json').read_text())
geo={}
for item in gm:
 if item['path'].endswith('elbase.csv'):
  for row in csv.reader(io.StringIO((ROOT/item['path']).read_text(encoding='utf-8-sig'))):
   if len(row)>5 and row[2:5]==['00','000','0000']:
    key=tuple(row[:2]);assert key not in geo or geo[key]==row[5];geo[key]=row[5]
history=defaultdict(list);rawrows={}
for item in am['members']:
 if item['path'] not in PINS or '2022-' not in item['archive_member'] or '/T' not in item['archive_member']:continue
 assert sha(ROOT/item['path'])==item['sha256']
 for lineno,row in enumerate(csv.reader(io.StringIO((ROOT/item['path']).read_text(encoding='utf-8-sig'))),1):
  assert len(row)==16
  rawrows[(item['path'],str(lineno))]=row
  h=dict(history_name=row[6],history_county=geo[tuple(row[:2])],history_district=str(int(row[2])),history_birth_roc=row[9],history_sex={'1':'男','2':'女'}.get(row[8],''),history_education=row[12],history_file=item['path'],history_row=str(lineno),history_archive_member=item['archive_member'],source_sha256=item['sha256'])
  history[(norm(h['history_county']),norm(h['history_name']))].append(h)
def iso_roc(s):
 if not re.fullmatch(r'\d{7}',s):return ''
 try:return datetime.date(int(s[:3])+1911,int(s[3:5]),int(s[5:])).isoformat()
 except ValueError:return ''
def update(key,field,value,evidence,status,note):
 mask=fields.candidate_key.eq(key)&fields.field.eq(field);assert mask.sum()==1
 next_source='帳號首頁與候選人官網交叉核對；貼文蒐集暫緩' if field in ['Facebook','Instagram','Threads'] else '2026公報／目前官方人物頁再核對'
 fields.loc[mask,['existing_value','evidence','status','note','next_source','updated_date']]=[value,evidence,status,note,next_source,'2026-10-19']
 mm=master.candidate_key.eq(key);assert mm.sum()==1;master.loc[mm,field+'_status']=status
review_index=review.set_index('candidate_key').to_dict('index')
camp_index=campaign.set_index('candidate_key').to_dict('index')
profile_keys=set(old_profiles.candidate_key)|set(campaign.candidate_key)
council=[];pending_history=[];verified_count=0
for p in master[master.office_type.str.contains('議員')].to_dict('records'):
 key=p['candidate_key'];a=review_index.get(key,{});c=camp_index.get(key,{})
 matches=history.get((norm(p['county_city']),norm(p['candidate_name'])),[])
 district=re.search(r'第(\d+)選舉區',p['electoral_district']);dn=str(int(district[1])) if district else ''
 supported=a.get('identity_link_supported')=='True' and a.get('history_birth_roc','')!=''
 dob='';sex='';hf='';hr='';status='尚無可連結的歷史紀錄'
 if supported:
  hf=a['history_file'];hr=a['history_row'];raw=rawrows[(hf,hr)]
  assert raw[6]==a['history_name'] and raw[9]==a['history_birth_roc'] and {'1':'男','2':'女'}.get(raw[8],'')==a['history_sex']
  dob=iso_roc(raw[9]);assert dob and sex=='';sex=a['history_sex'];assert sex in ['男','女']
  status='2022官方申報，先前身分連結審核有支持；非2026戶籍認定';verified_count+=1
  ev='https://db.cec.gov.tw/ | '+hf+':'+hr
  update(key,'birth_date',dob,ev,status,a['identity_review_result']+'；民國生日轉西元，來源年2022。')
  update(key,'sex',sex,ev,status,a['identity_review_result']+'；沿用2022名冊申報性別，保留年代。')
  for field in ['first_time_this_office','first_time_any_office']:update(key,field,'0',ev,'有2022議員參選紀錄，身分連結有支持','2022曾登記參選即支持非終生首次；不以當選與否判定，不推定2026仍在任。')
 elif matches:status='同名同縣市歷史命中，身分仍待核對；未填入已核實欄'
 if not supported:
  for h in matches:
   pending_history.append(dict(candidate_key=key,candidate_name=p['candidate_name'],county_city=p['county_city'],electoral_district=p['electoral_district'],same_district_number=str(h['history_district']==dn),birth_date_history_candidate=iso_roc(h['history_birth_roc']),identity_status=status,profile_evidence_present=str(key in profile_keys),**h))
 edu=c.get('education_raw','') or a.get('education_raw','');exp=c.get('experience_raw','') or a.get('experience_raw','')
 source=c.get('profile_url','') if c else a.get('profile_url','')
 fany='0' if supported else '';fthis='0' if supported else ''
 earlier=bg[bg.candidate_key.eq(key)&bg.field.eq('first_time_any_office')]
 if len(earlier):assert earlier.value_raw.eq('0').all();fany='0'
 earlier=bg[bg.candidate_key.eq(key)&bg.field.eq('first_time_this_office')]
 if len(earlier):assert earlier.value_raw.eq('0').all();fthis='0'
 council.append(dict(candidate_key=key,candidate_name=p['candidate_name'],county_city=p['county_city'],office_type=p['office_type'],electoral_district=p['electoral_district'],party_registration_raw=p['party_registration_raw'],birth_date_supported_history=dob,birth_year_supported_history=dob[:4] if dob else '',sex_supported_history=sex,birth_sex_evidence_status=status,history_source_file=hf,history_source_row=hr,history_identity_review=a.get('identity_review_result',''),history_match_count=len(matches),education_raw=edu,experience_raw=exp,background_source_url=source,moi_profile_url=a.get('profile_url',''),campaign_profile_url=c.get('profile_url',''),education_2022_declared=a.get('history_education_2022','') if supported else '',education_completion_status='保留來源原文，未認證學位；在學不當作已畢業',first_time_this_office_supported=fthis,first_time_any_office_supported=fany,incumbency_2026='',incumbency_status='未由歷史當選或履歷自動推定',updated_date='2026-10-19'))
assert verified_count==690
# Bring the separately preserved social evidence into the current field inventory.
for (key,plat),d in social.groupby(['candidate_key','platform'],sort=False):
 urls=' | '.join(dict.fromkeys(d.account_url));src=' | '.join(dict.fromkeys(d.source_url))
 update(key,plat,urls,src,'已保存來源連出的帳號入口；目標內容待核對','本輪僅找入口，貼文期間及貼文蒐集依學生指示暫緩；不是零貼文。')
council=pd.DataFrame(council);pending_history=pd.DataFrame(pending_history)
assert len(council)==1502 and council.candidate_key.is_unique
assert len(master)==1583 and master.candidate_key.is_unique
assert len(fields)==31660 and not fields.duplicated(['candidate_key','field']).any()
assert council.birth_date_supported_history.ne('').sum()==690 and council.sex_supported_history.ne('').sum()==690
assert council.incumbency_2026.eq('').all()
assert not fields[fields.field.str.startswith('posts_')].existing_value.ne('').any()
# New no-profile people remain a collection worklist, not verified identities.
unprofiled=council[~council.candidate_key.isin(profile_keys)]
assert len(unprofiled)==785
pending_no_profile=pending_history[pending_history.candidate_key.isin(unprofiled.candidate_key)]
outputs={'data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv':council,'data/processed/council_history_pending_identity_20261019.csv':pending_history,'data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv':master,'data/processed/snapshots/pre_source_expansion_20261019/candidate_fields_current.csv':fields}
verification={}
for path,frame in outputs.items():
 assert path not in PINS and (ROOT/path).is_relative_to(ROOT/'data/processed')
 frame.to_csv(ROOT/path,index=False,encoding='utf-8-sig');d=read(path)
 assert len(d)==len(frame) and list(d.columns)==list(frame.columns)
 assert len((ROOT/path).read_text(encoding='utf-8-sig').splitlines())==len(d)+1
 verification[path]={'rows':len(d),'sha256':sha(ROOT/path)}
summary={'council_total':len(council),'supported_history_birth_and_sex':690,'without_supported_birth_and_sex':int(council.birth_date_supported_history.eq('').sum()),'no_profile':785,'no_profile_with_2022_history_leads':int(pending_no_profile.candidate_key.nunique()),'pending_history_candidates':int(pending_history.candidate_key.nunique()),'pending_history_rows':len(pending_history),'non_first_any_office_supported':int(council.first_time_any_office_supported.eq('0').sum()),'non_first_council_supported':int(council.first_time_this_office_supported.eq('0').sum()),'social_candidate_platforms_linked':int(social[['candidate_key','platform']].drop_duplicates().shape[0]),'outputs':verification}
(ROOT/'data/code/current_inventory_20261019_verification.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2))
print(json.dumps(summary,ensure_ascii=False,indent=2))


{
  "council_total": 1502,
  "supported_history_birth_and_sex": 690,
  "without_supported_birth_and_sex": 812,
  "no_profile": 785,
  "no_profile_with_2022_history_leads": 157,
  "pending_history_candidates": 164,
  "pending_history_rows": 164,
  "non_first_any_office_supported": 694,
  "non_first_council_supported": 691,
  "social_candidate_platforms_linked": 132,
  "outputs": {
    "data/processed/snapshots/pre_source_expansion_20261019/council_background_20261019.csv": {
      "rows": 1502,
      "sha256": "2b63c4a399d39788c02d8ae09010ae6bdeaccda6842f41191a1eeffe4c89f81e"
    },
    "data/processed/council_history_pending_identity_20261019.csv": {
      "rows": 164,
      "sha256": "e88d1bca609d0a837dfc597e95d10d85a1dec12a5feaf8d542c6fc25388c143f"
    },
    "data/processed/snapshots/pre_source_expansion_20261019/candidate_master_current.csv": {
      "rows": 1583,
      "sha256": "fbb32bba0c26e5b266fe4c7710058f27353f86fdbc8d59f7c112d4a30451f37d"
    },
    "data/processed/snapshots

2026-10-20 維護：重複檔案改用單一保存路徑，固定指紋同步更新；議員登記CSV排版換行改為空格，政黨標籤不變。本次已獨立從raw表格重建並比對全部1,502筆及16欄；未重跑本Notebook或整條研究流程，既有執行輸出屬先前執行紀錄。